In [1]:
!pip install -q pandas numpy scikit-learn matplotlib seaborn joblib

import pandas as pd
import numpy as np
import re
import random
import unicodedata

from collections import Counter

RANDOM_STATE = 42

random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)

print("Среда успешно подготовлена!")
print("Pandas:", pd.__version__)
print("NumPy:", np.__version__)

Среда успешно подготовлена!
Pandas: 2.2.3
NumPy: 2.4.6


In [2]:
train_df = pd.read_csv("/content/sms_train.csv")
dev_df = pd.read_csv("/content/sms_dev.csv")

print("Train:", train_df.shape)
print("Dev:", dev_df.shape)

display(train_df.head())
display(dev_df.head())

Train: (200, 3)
Dev: (40, 3)


,id,text,label
0,TR001,Не переводите деньги незнакомым и не сообщайте...,ok
1,TR002,MegaCom: пополнение баланса на 300 сом. Спасибо!,ok
2,TR003,РСК Банк: зачисление 500 сом от Елена С.,ok
3,TR004,"Мама, у меня проблемы, срочно переведи 1500 со...",fraud
4,TR005,Beeline: подключите пакет «Безлимит» за 5000 с...,ok


,id,text,label
0,DV001,Bonus 500 s o m sizge! O! aktivdeshtiruu: http...,fraud
1,DV002,Код для входа в Бакай Банк Online: 457469. Не ...,ok
2,DV003,"Оптима Банк: ваш кредит одобрен, подробности в...",ok
3,DV004,"Заказ готов, з а б е р и т е в пункте выдачи. ...",ok
4,DV005,"Мама, у меня проблемы, срочно переведи 5000 со...",fraud


In [3]:
print("Пропуски:")
print(train_df.isna().sum())

print("\nДубликаты текста:")
print(train_df["text"].duplicated().sum())

print("\nКлассы:")
print(train_df["label"].value_counts())

print("\nПримеры FRAUD:")
display(train_df[train_df["label"] == "fraud"].head(5))

print("\nПримеры OK:")
display(train_df[train_df["label"] == "ok"].head(5))

Пропуски:
id       0
text     0
label    0
dtype: int64

Дубликаты текста:
0

Классы:
label
ok       100
fraud    100
Name: count, dtype: int64

Примеры FRAUD:


,id,text,label
3,TR004,"Мама, у меня проблемы, срочно переведи 1500 со...",fraud
8,TR009,Служба безопасности MBANK. Зафиксирована подоз...,fraud
11,TR012,Сиздин номериңиз 24 сааттан кийин өчүрүлөт. Те...,fraud
13,TR014,РСК Банк: картаңыз бөгөттөлдү. Тез арада SMS'т...,fraud
14,TR015,Куттуктайбыз! Сиз 50 000 сом уттуңуз. Алуу үчү...,fraud



Примеры OK:


,id,text,label
0,TR001,Не переводите деньги незнакомым и не сообщайте...,ok
1,TR002,MegaCom: пополнение баланса на 300 сом. Спасибо!,ok
2,TR003,РСК Банк: зачисление 500 сом от Елена С.,ok
4,TR005,Beeline: подключите пакет «Безлимит» за 5000 с...,ok
5,TR006,Сиздин 10 ГБ пакетиңиз uйгuлuктүү тут@штырылды...,ok


In [4]:
import re
import unicodedata

# Защищённые слова
PROTECTED_WORDS = {
    "sim", "sms", "url", "pin",
    "megacom", "beeline", "mbank", "kicb"
}

# Кириллические символы, похожие на латинские
CYRILLIC_TO_LATIN = str.maketrans({
    "а": "a", "с": "c", "е": "e",
    "о": "o", "р": "p", "х": "x",
    "у": "y", "к": "k", "м": "m",
    "т": "t", "в": "b", "н": "h"
})

# Замена цифр и символов в кириллических словах
CYRILLIC_OBFUSCATION = str.maketrans({
    "0": "о", "3": "з", "4": "ч",
    "6": "б", "@": "а", "$": "с"
})


def normalize_word(word: str) -> str:

    if word in PROTECTED_WORDS:
        return word

    has_latin = bool(re.search(r"[a-z]", word))
    has_cyrillic = bool(re.search(r"[а-яёөүң]", word))

    if has_latin:
        word = word.translate(CYRILLIC_TO_LATIN)
        word = word.replace("@", "a")
        word = word.replace("3", "e")
        word = word.replace("1", "i")

    elif has_cyrillic:
        word = word.translate(CYRILLIC_OBFUSCATION)

    return word


def normalize_text(text: str) -> str:

    if not isinstance(text, str):
        return ""

    text = unicodedata.normalize("NFKC", text).lower()

    # Сохраняем ссылки
    urls = []

    def save_url(match):
        urls.append(match.group(0))
        return f" URLTOKEN{len(urls) - 1}END "

    text = re.sub(
        r"(?:https?://|www\.)[^\s]+",
        save_url,
        text
    )

    # Сохраняем домены
    domains = []

    def save_domain(match):
        domains.append(match.group(0))
        return f" DOMAINTOKEN{len(domains) - 1}END "

    text = re.sub(
        r"(?<![\w@])(?:[a-z0-9-]+\.)+[a-z]{2,}(?![\w-])",
        save_domain,
        text
    )

    # Сохраняем телефонные номера
    phones = []

    def save_phone(match):
        phones.append(match.group(0))
        return f" PHONETOKEN{len(phones) - 1}END "

    text = re.sub(
        r"(?<!\w)(?:\+?\d[\d\s()\-]{6,}\d)(?!\w)",
        save_phone,
        text
    )

    # Убираем разделители внутри слов
    text = re.sub(
        r"(?<=[а-яёөүңa-z])[._*]+(?=[а-яёөүңa-z])",
        "",
        text
    )

    text = re.sub(
        r"\b(sim|sms|url|pin)-(?=[а-яёөүң])",
        r"\1 ",
        text
    )

    text = re.sub(
        r"(?<=[а-яёөүңa-z])-(?=[а-яёөүңa-z])",
        "",
        text
    )

    # Объединяем раздельные буквы
    text = re.sub(
        r"(?<!\S)(?:[а-яёөүңa-z]\s+){2,}[а-яёөүңa-z](?=\s+[а-яёөүңa-z]{2,}|[^\w\s]|$)",
        lambda m: re.sub(r"\s+", "", m.group(0)),
        text
    )

    # Исправляем написание слов
    def fix_word(match):
        word = match.group(0)

        if word in PROTECTED_WORDS:
            return word

        return normalize_word(word)

    text = re.sub(
        r"[а-яёөүңa-z0-9@$]+",
        fix_word,
        text
    )

    # Возвращаем ссылки
    for i, url in enumerate(urls):
        text = text.replace(f"URLTOKEN{i}END", url)

    # Возвращаем домены
    for i, domain in enumerate(domains):
        text = text.replace(f"DOMAINTOKEN{i}END", domain)

    # Возвращаем телефоны
    for i, phone in enumerate(phones):
        text = text.replace(f"PHONETOKEN{i}END", phone)

    text = re.sub(r"\s+", " ", text).strip()

    return text

In [5]:
tests = [
    "V@sh sch3t zablokirovan",
    "S1zdin esebiniz bloktoldu",
    "Vаsh bаnk",
    "Ваш банк требует подтвердить пароль",
    "Ср0чн0 отправьте к0д",
    "к о д отправьте срочно",
    "SIM-карта",
    "MBank: ваш баланс пополнен",
    "https://example.com",
    "Для регистрации посетите example.com",
    "pay.mbank.kg",
    "fake-mbank.kg",
    "mbank.kg.example",
    "+996 555 123 456"
]

for item in tests:
    print("ORIGINAL:  ", item)
    print("NORMALIZED:", normalize_text(item))
    print("-" * 60)

ORIGINAL:   V@sh sch3t zablokirovan
NORMALIZED: vash schet zablokirovan
------------------------------------------------------------
ORIGINAL:   S1zdin esebiniz bloktoldu
NORMALIZED: sizdin esebiniz bloktoldu
------------------------------------------------------------
ORIGINAL:   Vаsh bаnk
NORMALIZED: vash bank
------------------------------------------------------------
ORIGINAL:   Ваш банк требует подтвердить пароль
NORMALIZED: ваш банк требует подтвердить пароль
------------------------------------------------------------
ORIGINAL:   Ср0чн0 отправьте к0д
NORMALIZED: срочно отправьте код
------------------------------------------------------------
ORIGINAL:   к о д отправьте срочно
NORMALIZED: код отправьте срочно
------------------------------------------------------------
ORIGINAL:   SIM-карта
NORMALIZED: sim карта
------------------------------------------------------------
ORIGINAL:   MBank: ваш баланс пополнен
NORMALIZED: mbank: ваш баланс пополнен
-----------------------------

In [6]:
import re

def extract_features(text: str) -> dict:
    text = normalize_text(text)

    features = {
        # 1. Наличие ссылки
        "has_url": int(bool(re.search(
            r"(https?://|www\.|[a-z0-9-]+\.[a-z]{2,})",
            text
        ))),

        # 2. Запрос кода или пароля
        "asks_code_password": int(bool(re.search(
            r"(код|кодду|кодун|парол[ья]|сырсөз|сырсөзүңүз|pin|sms.?код)",
            text
        ))),

        # 3. Срочность
        "urgency": int(bool(re.search(
            r"(срочно|немедленно|быстро|сейчас|тез|дароо|азыр)",
            text
        ))),

        # 4. Угроза блокировки
        "blocking_threat": int(bool(re.search(
            r"(заблокирован|блокировка|блоктолот|блоктолду|бөгөттөлдү|бөгөттөлөт|жабылат|жабылды)",
            text
        ))),

        # 5. Выигрыш или приз
        "prize": int(bool(re.search(
            r"(bonus|бонус|выиграл|выиграли|победител\w*|приз|подарок|байге|уттуңуз|утту|жеңүүчү|утуп алдыңыз)",
            text
        ))),

        # 6. Банк или оператор
        "bank_operator": int(bool(re.search(
            r"(банк|bank|mbank|kicb|рск|оптима|megacom|beeline|элкарт)",
            text
        ))),

        # 7. Перевод денег
        "money_transfer": int(bool(re.search(
            r"(переведите|перевод|отправьте деньги|пополните счет|акча котор|акча жөнөт|акчаңызды)",
            text
        ))),

        # 8. Просьба перезвонить
        "callback_request": int(bool(re.search(
            r"(перезвоните|позвоните|обратитесь по номеру|чалгыла|кайра чалыңыз)",
            text
        ))),

        # 9. Просьба отправить или предоставить информацию
        "send_request": int(bool(re.search(
            r"(отправьте|сообщите|введите|подтвердите|подтвердить|активируйте|активировать|активдештир|aktivdeshtir|aktivirovat|жөнөтүңүз|жибериңиз|жазыңыз|бериңиз|киргизиңиз)",
            text
        ))),

        # 10. Упоминание аккаунта или карты
        "account_card": int(bool(re.search(
            r"(аккаунт|сч[её]т|карта|card|эсебиңиз|эсеп|картаңыз|пароль|парол[ья]|сырсөз|кабинет)",
            text
        ))),

        # 11. Просьба перейти по ссылке
        "link_action": int(bool(re.search(
            r"(перейдите по ссылке|перейти по ссылке|шилтемеге өт|шилтемени бас|ссылка боюнча өт)",
            text
        )))
    }

    return features

In [7]:
test_messages = [
    "Ваш счёт заблокирован! Срочно отправьте SMS-код.",
    "Вы выиграли приз! Переведите 300 сом для получения.",
    "Для регистрации посетите example.com",
    "Ваш баланс успешно пополнен на 300 сом.",
    "Сиздин эсебиңиз бөгөттөлдү. SMS кодду жөнөтүңүз.",
    "Bonus 500 som! Aktivdeshtiruu: https://kicb-online.example/card"
]

for message in test_messages:
    print("\nSMS:", message)
    print("Нормализация:", normalize_text(message))
    print("Признаки:", extract_features(message))
    print("-" * 80)


SMS: Ваш счёт заблокирован! Срочно отправьте SMS-код.
Нормализация: ваш счёт заблокирован! срочно отправьте sms код.
Признаки: {'has_url': 0, 'asks_code_password': 1, 'urgency': 1, 'blocking_threat': 1, 'prize': 0, 'bank_operator': 0, 'money_transfer': 0, 'callback_request': 0, 'send_request': 1, 'account_card': 1, 'link_action': 0}
--------------------------------------------------------------------------------

SMS: Вы выиграли приз! Переведите 300 сом для получения.
Нормализация: вы выиграли приз! переведите 300 сом для получения.
Признаки: {'has_url': 0, 'asks_code_password': 0, 'urgency': 0, 'blocking_threat': 0, 'prize': 1, 'bank_operator': 0, 'money_transfer': 1, 'callback_request': 0, 'send_request': 0, 'account_card': 0, 'link_action': 0}
--------------------------------------------------------------------------------

SMS: Для регистрации посетите example.com
Нормализация: для регистрации посетите example.com
Признаки: {'has_url': 1, 'asks_code_password': 0, 'urgency': 0, 'b

In [8]:
example_features = extract_features(
    "Ваш счёт заблокирован! Срочно отправьте SMS-код."
)

print("Количество признаков:", len(example_features))
print("Названия признаков:", list(example_features.keys()))

Количество признаков: 11
Названия признаков: ['has_url', 'asks_code_password', 'urgency', 'blocking_threat', 'prize', 'bank_operator', 'money_transfer', 'callback_request', 'send_request', 'account_card', 'link_action']


In [9]:
train_df["normalized_text"] = train_df["text"].apply(normalize_text)
dev_df["normalized_text"] = dev_df["text"].apply(normalize_text)

X_train = train_df["normalized_text"]
y_train = train_df["label"]

X_dev = dev_df["normalized_text"]
y_dev = dev_df["label"]

print("Обучающих сообщений:", len(X_train))
print("Проверочных сообщений:", len(X_dev))

print("\nРаспределение обучающих классов:")
print(y_train.value_counts())

Обучающих сообщений: 200
Проверочных сообщений: 40

Распределение обучающих классов:
label
ok       100
fraud    100
Name: count, dtype: int64


In [10]:
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import FunctionTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction import DictVectorizer


def extract_features_for_model(texts):
    return [
        extract_features(text)
        for text in texts
    ]


numeric_features = Pipeline([
    ("extract", FunctionTransformer(
        extract_features_for_model,
        validate=False
    )),
    ("vectorize", DictVectorizer())
])

print("Обработка числовых признаков подготовлена!")

Обработка числовых признаков подготовлена!


In [11]:
features = FeatureUnion([
    ("char", TfidfVectorizer(
        analyzer="char",
        ngram_range=(2, 5),
        max_features=30000,
        sublinear_tf=True
    )),

    ("word", TfidfVectorizer(
        analyzer="word",
        ngram_range=(1, 2),
        max_features=15000,
        sublinear_tf=True
    )),

    ("numeric", numeric_features)
])

print("Все три вида признаков объединены!")

Все три вида признаков объединены!


In [12]:
model = Pipeline([
    ("features", features),

    ("classifier", LogisticRegression(
        C=1.0,
        max_iter=2000,
        class_weight="balanced",
        random_state=42
    ))
])

model.fit(X_train, y_train)

print("Модель успешно обучена!")

Модель успешно обучена!


In [13]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    confusion_matrix,
    classification_report
)

In [14]:
y_pred = model.predict(X_dev)

In [15]:
print("Accuracy:", accuracy_score(y_dev, y_pred))

print(
    "Fraud Precision:",
    precision_score(y_dev, y_pred, pos_label="fraud")
)

print(
    "Fraud Recall:",
    recall_score(y_dev, y_pred, pos_label="fraud")
)

print("\nClassification Report:")
print(classification_report(y_dev, y_pred))

Accuracy: 1.0
Fraud Precision: 1.0
Fraud Recall: 1.0

Classification Report:
              precision    recall  f1-score   support

       fraud       1.00      1.00      1.00        20
          ok       1.00      1.00      1.00        20

    accuracy                           1.00        40
   macro avg       1.00      1.00      1.00        40
weighted avg       1.00      1.00      1.00        40



In [16]:
cm = confusion_matrix(
    y_dev,
    y_pred,
    labels=["ok", "fraud"]
)

print("Матрица ошибок:")
print(cm)

Матрица ошибок:
[[20  0]
 [ 0 20]]


In [17]:
errors = dev_df.copy()

errors["prediction"] = y_pred
errors["correct"] = errors["label"] == errors["prediction"]

print("Количество ошибок:", (~errors["correct"]).sum())

display(
    errors[errors["correct"] == False][
        ["text", "label", "prediction"]
    ]
)

Количество ошибок: 0


,text,label,prediction


In [18]:
def get_final_label(fraud_probability):
    if fraud_probability >= 0.70:
        return "high"
    elif fraud_probability >= 0.40:
        return "medium"
    else:
        return "low"

In [19]:
test_probabilities = [0.15, 0.45, 0.65, 0.75, 0.95]

for p in test_probabilities:
    print(p, "->", get_final_label(p))

0.15 -> low
0.45 -> medium
0.65 -> medium
0.75 -> high
0.95 -> high


In [20]:
def predict_sms(test_df):
    normalized = test_df["text"].apply(normalize_text)

    probabilities = model.predict_proba(normalized)

    fraud_index = list(model.classes_).index("fraud")
    fraud_probs = probabilities[:, fraud_index]

    predictions = [
        get_final_label(p)
        for p in fraud_probs
    ]

    return pd.DataFrame({
        "id": test_df["id"],
        "label": predictions
    })

In [21]:
demo_df = pd.DataFrame({
    "id": [1, 2, 3, 4, 5],
    "text": [
        "Ваш баланс успешно пополнен.",
        "Срочно отправьте SMS-код, иначе счёт заблокируют.",
        "Вы выиграли приз! Получите деньги по ссылке.",
        "Завтра состоится собрание.",
        "Сиздин эсебиңиз бөгөттөлдү. Кодду жөнөтүңүз."
    ]
})

display(demo_df)

,id,text
0,1,Ваш баланс успешно пополнен.
1,2,"Срочно отправьте SMS-код, иначе счёт заблокируют."
2,3,Вы выиграли приз! Получите деньги по ссылке.
3,4,Завтра состоится собрание.
4,5,Сиздин эсебиңиз бөгөттөлдү. Кодду жөнөтүңүз.


In [22]:
result = predict_sms(demo_df)

display(result)

print("\nРаспределение меток:")
print(result["label"].value_counts())

,id,label
0,1,low
1,2,high
2,3,high
3,4,low
4,5,high



Распределение меток:
label
high    3
low     2
Name: count, dtype: int64


In [23]:
import joblib

joblib.dump(model, "/content/sms_fraud_model.pkl")

print("Модель успешно сохранена!")

Модель успешно сохранена!


In [25]:
loaded_model = joblib.load("/content/sms_fraud_model.pkl")

print("Модель успешно загружена!")
print("Классы модели:", loaded_model.classes_)

Модель успешно загружена!
Классы модели: ['fraud' 'ok']


In [26]:
test_text = ["Поздравляем! Вы выиграли приз. Перейдите по ссылке для получения."]

normalized = [normalize_text(text) for text in test_text]

probabilities = loaded_model.predict_proba(normalized)

fraud_index = list(loaded_model.classes_).index("fraud")

print("Вероятность мошенничества:", probabilities[0][fraud_index])

Вероятность мошенничества: 0.9209912856293182


In [27]:
from google.colab import files

files.download("/content/sms_fraud_model.pkl")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>